In [ ]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

In [9]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/facebook_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    facebook_posts = json.load(f)

In [10]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for fb_post in facebook_posts:
    post_id = fb_post["post_id"]
    all_post_ids.append(post_id)
    text = (fb_post["content"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["post_id"]: post for post in facebook_posts}

Total posts: 123000
Total posts with a theme: 80322
Found environmental posts: 12451
Found infrastructure posts: 35918
Found housing posts: 13527
Found economic posts: 40351
Found life quality posts: 27041
Found aesthetic posts: 12608
Found government posts: 11506


In [ ]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "number of shares", "page followers", "is page verified", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of comments", "number of shares", "page followers", "is page verified", "is paid partnership", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_likes = []
    post_comment_numbers = []
    post_share_numbers = []
    post_page_followers = []
    post_is_page_verified = []
    post_is_paid_partnership = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["content"])
        post_dates.append(post["date_posted"])
        post_likes.append(post["likes"])
        post_comment_numbers.append(post["num_comments"])
        post_share_numbers.append(post["num_shares"])
        post_page_followers.append(post["page_followers"])
        post_is_page_verified.append(post["page_is_verified"])
        post_is_paid_partnership.append(post["is_sponsored"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["likes"] = post_likes
    df1["number of comments"] = post_comment_numbers
    df1["number of shares"] = post_share_numbers
    df1["page followers"] = post_page_followers
    df1["is page verified"] = post_is_page_verified
    df1["is paid partnership"] = post_is_paid_partnership

    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)


# Convert to proper dtypes
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "likes": "int64",
    "number of comments": "int64",
    "number of shares": "int64",
    "page followers": "int64",
    "is page verified": "bool",
    "is paid partnership": "bool",
    "environment": "bool",
    "infrastructure": "bool",
    "housing": "bool",
    "economy": "bool",
    "life quality": "bool",
    "aesthetics": "bool",
    "government": "bool",
})

len(posts['ids'])

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_57896/404807702.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)


153402

In [12]:
def remove_links(text):
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [13]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

80322

In [ ]:
company_terms = ["AWS","Amazon","Google","Microsoft","Azure","Meta","Oracle","Equinix","Digital Realty","IBM","Facebook","Apple","QTS","Vantage","CyrusOne","CoreSite"]

for c in company_terms:
    posts[c] = posts["text"].str.contains(rf'\b{re.escape(c)}\b', case=False, na=False)

posts_with_company = posts[posts[company_terms].any(axis=1)]
posts_with_company.head()

In [14]:
posts.head()

,ids,text,date,likes,number of comments,number of shares,page followers,is page verified,is paid partnership,environment,infrastructure,housing,economy,life quality,aesthetics,government
0,1000003621927424,🔧🎉 Happy Manufacturing Day from Boundary Elect...,2024-10-04T15:22:18.000Z,25.0,0,2,600.0,False,False,False,True,False,True,False,False,False
1,1000006858576582,The new EDP Distribution website is now live.\...,2024-07-31T14:50:59.000Z,2.0,0,0,110.0,False,False,False,False,True,False,False,False,False
2,1000016557087375,"European Contracts Manager, within the Fire Se...",2020-08-25T16:09:01.000Z,1.0,0,2,541.0,False,False,False,False,False,True,False,False,False
3,1000032582123564,🌟 GNSA is thrilled to announce our collaborati...,2024-06-12T07:39:07.000Z,2.0,0,0,467.0,False,False,False,False,False,True,False,True,False
4,1000051012170693,En el proyecto europeo 8ra (también conocido c...,2025-01-09T09:21:39.000Z,0.0,0,0,13000.0,False,False,False,False,False,False,True,False,False


In [35]:
# def convert_dates(utc_date):
#     date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
#     return date

# posts['date'] = posts['date'].apply(convert_dates)
# posts.head(10)

In [ ]:
# Make sure your pipeline returns all class scores, not just the top label
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    padding=True,
    max_length=512,
    top_k=None
)

tokenizer = classifier.tokenizer

def chunk_text(text, max_tokens=450):
    """
    Split one post into chunks small enough for RoBERTa.
    450 leaves room for special tokens and avoids hitting the 512 limit.
    """
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return [""]

    chunks = []

    for i in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[i:i + max_tokens]
        chunk = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk)

    return chunks


def aggregate_chunk_results(chunk_results):
    """
    Average sentiment scores across chunks.
    """
    scores = {
        "negative": [],
        "neutral": [],
        "positive": []
    }

    for result in chunk_results:
        for item in result:
            label = item["label"].lower()
            scores[label].append(item["score"])

    avg_scores = {
        label: float(np.mean(values)) if values else 0.0
        for label, values in scores.items()
    }

    final_label = max(avg_scores, key=avg_scores.get)

    return {
        "label": final_label,
        "score": avg_scores[final_label],
        "scores": avg_scores,
        "n_chunks": len(chunk_results)
    }

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
texts= posts["text"].fillna("").astype(str).tolist()
all_post_results = []

for text in tqdm(texts):
    chunks = chunk_text(text)

    chunk_results = classifier(
        chunks,
        truncation=True,
        padding=True,
        max_length=512,
        batch_size=32
    )

    post_result = aggregate_chunk_results(chunk_results)
    all_post_results.append(post_result)

posts["roberta"] = all_post_results

In [ ]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])
posts["label"] = posts["name"]

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,votes,number of comments,environment,infrastructure,housing,economy,life quality,aesthetics,government,label,degree
0,10s4xtb,People who have left NOVA or planning to leave...,1.675382e+09,129,308.0,False,False,False,False,False,False,True,negative,0.544824
1,117zom7,Is staying in nova even worth it? I posted thi...,1.676973e+09,8,22.0,False,False,True,True,False,False,True,negative,0.649185
2,1341j3h,Anyone have HD or 4k (drone) footage of Ashbur...,1.682885e+09,0,8.0,False,False,False,False,True,False,True,negative,0.794359
3,14gh0c1,Proposed data center in Chantilly stirs up con...,1.687473e+09,4,7.0,True,True,True,False,False,False,False,negative,0.910441
4,152m2sz,Microsoft acquires 14-acre site in Sterling fo...,1.689648e+09,86,43.0,False,False,True,False,True,False,False,negative,0.906577
5,158wywd,Can someone tell me whats going on on barriste...,1.690257e+09,0,7.0,False,False,True,False,True,False,False,negative,0.690163
6,15cwyr6,Aren't the Loudon datacenters actually awesome...,1.690649e+09,413,246.0,False,True,True,True,True,False,False,negative,0.418970
7,171nn12,"Looking for cable tech jobs Hello, I currentl...",1.696626e+09,0,4.0,False,False,True,True,False,False,False,negative,0.537280
8,1al34nd,Need help with sources and talking points AGAI...,1.707313e+09,0,27.0,False,False,True,False,False,False,False,negative,0.721406
9,1bdp55a,Fairfax County considers enhancing data center...,1.710329e+09,6,7.0,True,False,True,False,True,True,True,negative,0.924399


In [41]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: -1.0
Average sentiment of infrastructural posts: -1.0
Average sentiment of housing-related posts: -1.0
Average sentiment of economic posts: -1.0
Average sentiment of life-quality-related posts: -1.0
Average sentiment of aesthetics-related posts: -1.0
Average sentiment of governmental posts: -1.0


In [ ]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for all of facebook: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


In [ ]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.0
Percent of posts that are negative: 1.0
Percent of posts that are positive: 0.0
Percent of negative posts that are really negative: 0.4957983193277311


In [ ]:
def avg_sentiment_calculation(dataset):
    pos = dataset.loc[dataset["label"] == "positive", "degree"].sum()
    neg = dataset.loc[dataset["label"] == "negative", "degree"].sum()
    total = dataset["degree"].sum()
    if total != 0:
        return (pos-neg)/total
    else:
        return 0

# Extract year from the date string
posts['year'] = pd.to_datetime(posts['date']).dt.year

# Create datasets for each year (efficient approach using a dictionary)
year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

# Access them like:
posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]

for i in range(2010, 2027):
    print(f"Number of posts from {i}: ", len(year_datasets[i]))
    if(avg_sentiment_calculation(year_datasets[i]) != 0):
        print(f"Average sentiment for facebook posts from {i}: ", avg_sentiment_calculation(year_datasets[i]))

In [ ]:
def sentiment_by_company(company1, company2=None):
    company_posts = posts[
        (posts[company1] == True) | (posts[company2] == True) if company2 else (posts[company1] == True)
    ]
    return avg_sentiment_calculation(company_posts)

print("Number of Amazon-related posts:", len(posts[posts["AWS"] == True]) + len(posts[posts["Amazon"] == True]))
print("Average sentiment of AWS-related posts:", sentiment_by_company("AWS", "Amazon"))
print("Number of Google-related posts:", len(posts[posts["Google"] == True]))
print("Average sentiment of Google-related posts:", sentiment_by_company("Google"))
print("Number of Microsoft-related posts:", len(posts[posts["Microsoft"] == True]) + len(posts[posts["Azure"] == True]))
print("Average sentiment of Microsoft-related posts:", sentiment_by_company("Microsoft", "Azure"))
print("Number of Meta-related posts:", len(posts[posts["Meta"] == True]) + len(posts[posts["Facebook"] == True]))
print("Average sentiment of Meta-related posts:", sentiment_by_company("Meta"))
print("Number of Oracle-related posts:", len(posts[posts["Oracle"] == True]))
print("Average sentiment of Oracle-related posts:", sentiment_by_company("Oracle"))
print("Number of Equinix-related posts:", len(posts[posts["Equinix"] == True]))
print("Average sentiment of Equinix-related posts:", sentiment_by_company("Equinix"))
print("Number of Digital Realty-related posts:", len(posts[posts["Digital Realty"] == True]))
print("Average sentiment of Digital Realty-related posts:", sentiment_by_company("Digital Realty"))
print("Number of IBM-related posts:", len(posts[posts["IBM"] == True]))
print("Average sentiment of IBM-related posts:", sentiment_by_company("IBM"))
print("Number of Apple-related posts:", len(posts[posts["Apple"] == True]))
print("Average sentiment of Apple-related posts:", sentiment_by_company("Apple"))
print("Number of QTS-related posts:", len(posts[posts["QTS"] == True]))
print("Average sentiment of QTS-related posts:", sentiment_by_company("QTS"))
print("Number of Vantage-related posts:", len(posts[posts["Vantage"] == True]))
print("Average sentiment of Vantage-related posts:", sentiment_by_company("Vantage"))
print("Number of CyrusOne-related posts:", len(posts[posts["CyrusOne"] == True]))
print("Average sentiment of CyrusOne-related posts:", sentiment_by_company("CyrusOne"))
print("Number of CoreSite-related posts:", len(posts[posts["CoreSite"] == True]))
print("Average sentiment of CoreSite-related posts:", sentiment_by_company("CoreSite"))

In [ ]:
def analysis_of_viral_posts(min):
    viral_posts = posts.loc[posts["likes"] >= min]
    print("What's considered viral: posts with over", min, "likes")
    print("Number of viral posts:", len(viral_posts))
    print("Average sentiment of viral posts:", avg_sentiment_calculation(viral_posts))
    print("Average sentiment of non-viral posts:", avg_sentiment_calculation(posts.loc[posts["likes"] < min]))
    print("Amount percent of viral posts that are polar (degree) > 0.70: ", len(viral_posts.loc[viral_posts["degree"] > 0.70])/len(viral_posts))

analysis_of_viral_posts(50)
print("\n")
analysis_of_viral_posts(100)